In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from ultralytics import YOLO
from collections import deque
from IPython.display import HTML, display
from base64 import b64encode
import os

# --- 1. CONFIGURATION PARAMETERS ---
# ==============================================================================
# --- Model & File Paths ---
MODEL_NAME = 'yolov8x.pt'
VIDEO_PATH = '/content/test.mp4'
OUTPUT_PATH = "/content/final_football_analysis_v9_refactored.mp4"

# --- Detection & Tracking Thresholds ---
CONFIDENCE_THRESHOLD = 0.3
CLASSES_TO_TRACK = [32]  # Class ID for 'sports ball' in the COCO dataset

# --- Analysis & State Logic Thresholds ---
HISTORY_LENGTH = 60  # Number of frames to store for position history
STATIONARY_SPEED_THRESHOLD = 2.0  # Pixels per frame; below this, a ball is 'STATIONARY'
POSSESSION_PROXIMITY_THRESHOLD = 120  # Max distance in pixels for a player to possess the ball

# --- Track Stitching & Re-identification Logic ---
REACQUIRE_THRESHOLD = 100  # Max distance to re-link a lost track
MAX_LOST_FRAMES = 30       # Max frames to remember a lost track before discarding
ZONE_OF_CONTROL_RADIUS = 150 # Radius around player's feet for priority re-linking

# --- 2. DATA STRUCTURES ---
# ==============================================================================
class Ball:
    """A class to represent and track the state of a single football."""
    def __init__(self, track_id, initial_pos):
        self.id = track_id
        self.positions = deque([initial_pos], maxlen=HISTORY_LENGTH)
        self.state = 'STATIONARY'
        self.velocity = np.array([0.0, 0.0])
        self.frames_stationary = 0

    def update(self, centroid):
        """Updates the ball's position, velocity, and state."""
        self.positions.append(centroid)
        if len(self.positions) > 5:
            # Calculate velocity over a few frames for stability
            self.velocity = (np.array(self.positions[-1]) - np.array(self.positions[-5])) / 4.0
        elif len(self.positions) > 1:
            self.velocity = np.array(self.positions[-1]) - np.array(self.positions[-2])

        # Update stationary counter
        if self.speed < STATIONARY_SPEED_THRESHOLD:
            self.frames_stationary += 1
        else:
            self.frames_stationary = 0

    @property
    def speed(self):
        """Calculates the scalar speed of the ball."""
        return np.linalg.norm(self.velocity)

    @property
    def last_known_position(self):
        """Returns the most recent centroid of the ball."""
        return self.positions[-1] if self.positions else None

    @property
    def is_stationary_marker(self):
        """Determines if the ball is a stationary marker (not the action ball)."""
        # A ball is a marker if it has been stationary for a significant time.
        return self.frames_stationary > 30

# --- 3. CORE FUNCTIONS ---
# ==============================================================================

def process_player_pose(frame, pose_model, frame_width, frame_height):
    """
    Processes a single frame to detect the player's pose and find their feet.
    Args:
        frame: The input video frame.
        pose_model: The initialized MediaPipe Pose model.
        frame_width (int): Width of the video frame.
        frame_height (int): Height of the video frame.
    Returns:
        A tuple containing the pose results and a list of foot centroids.
    """
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pose_results = pose_model.process(rgb_frame)
    player_feet_centroids = []
    if pose_results.pose_landmarks:
        landmarks = pose_results.pose_landmarks.landmark
        left_ankle = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE]
        right_ankle = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE]
        if left_ankle.visibility > 0.5 and right_ankle.visibility > 0.5:
            left_foot_pos = (int(left_ankle.x * frame_width), int(left_ankle.y * frame_height))
            right_foot_pos = (int(right_ankle.x * frame_width), int(right_ankle.y * frame_height))
            player_feet_centroids.extend([left_foot_pos, right_foot_pos])
    return pose_results, player_feet_centroids


def update_ball_states(tracked_balls, player_feet_centroids):
    """
    Updates the state of each tracked ball (STATIONARY, IN_POSSESSION, etc.).
    """
    for ball in tracked_balls.values():
        if ball.speed < STATIONARY_SPEED_THRESHOLD:
            ball.state = 'STATIONARY'
        else:
            is_possessed = False
            if player_feet_centroids:
                dist_to_foot = min([np.linalg.norm(ball.last_known_position - np.array(foot)) for foot in player_feet_centroids])
                if dist_to_foot < POSSESSION_PROXIMITY_THRESHOLD:
                    is_possessed = True
            ball.state = 'IN_POSSESSION' if is_possessed else 'ROLLING_AWAY'


def draw_annotations(frame, tracked_balls, boxes_map, pose_results, player_feet_centroids):
    """
    Draws all annotations on the frame (skeletons, ball trackers, boxes).
    """
    # Draw player pose skeleton
    if pose_results.pose_landmarks:
        mp_drawing.draw_landmarks(frame, pose_results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

    # Draw Zone of Control for debugging
    if player_feet_centroids:
        for foot_pos in player_feet_centroids:
            cv2.circle(frame, foot_pos, ZONE_OF_CONTROL_RADIUS, (255, 255, 0), 1, cv2.LINE_AA)

    # Draw annotations for each ball
    for ball in tracked_balls.values():
        pos = ball.last_known_position
        if pos is None: continue
        int_pos = (int(pos[0]), int(pos[1]))

        # --- DYNAMIC VISUALIZATION LOGIC ---
        if ball.is_stationary_marker:
            # Draw a bounding box for stationary marker balls
            box = boxes_map.get(ball.id)
            if box is not None:
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label = f"ID:{ball.id} (Marker)"
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        else:
            # Draw a tracking point for the action ball
            label = f"ID:{ball.id} | {ball.state}"
            color = {'IN_POSSESSION': (0, 0, 255), 'STATIONARY': (255, 0, 0), 'ROLLING_AWAY': (0, 165, 255)}[ball.state]
            cv2.circle(frame, int_pos, 10, color, -1)
            cv2.putText(frame, label, (int_pos[0] + 15, int_pos[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)


# --- 4. MAIN EXECUTION ---
# ==============================================================================
def main():
    """
    Main function to run the entire video processing pipeline.
    """
    # --- Initialization ---
    print(f"Loading YOLO model: {MODEL_NAME}...")
    model = YOLO(MODEL_NAME)

    global mp_pose, mp_drawing
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(static_image_mode=False, model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5)

    if not os.path.exists(VIDEO_PATH):
        print(f"❌ Error: Video file not found at '{VIDEO_PATH}'.")
        return

    cap = cv2.VideoCapture(VIDEO_PATH)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

    tracked_balls = {}
    lost_tracks = {}
    frame_count = 0

    print("✅ Initialization complete. Starting video processing...")

    # --- Main Processing Loop ---
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        annotated_frame = frame.copy()

        # 1. Process Player Pose
        pose_results, player_feet_centroids = process_player_pose(annotated_frame, pose_model, frame_width, frame_height)

        # 2. Run YOLO Tracker
        track_results = model.track(frame, persist=True, conf=CONFIDENCE_THRESHOLD, classes=CLASSES_TO_TRACK, verbose=False)

        final_track_ids = []
        boxes_map = {}
        if track_results[0].boxes.id is not None:
            raw_track_ids = track_results[0].boxes.id.int().cpu().tolist()
            raw_boxes = track_results[0].boxes.xyxy.cpu().tolist()
            final_track_ids = list(raw_track_ids) # Start with the raw list
            boxes_map = {tid: box for tid, box in zip(raw_track_ids, raw_boxes)}

            # 3. Intelligent Track Stitching Logic
            new_ids = set(final_track_ids) - set(tracked_balls.keys())

            for new_id in list(new_ids):
                new_centroid = np.array([ (boxes_map[new_id][0] + boxes_map[new_id][2]) / 2, (boxes_map[new_id][1] + boxes_map[new_id][3]) / 2 ])
                best_match_id, min_dist = None, REACQUIRE_THRESHOLD

                # Check against lost tracks
                for lost_id, data in lost_tracks.items():
                    dist = np.linalg.norm(new_centroid - data['last_pos'])
                    if dist < min_dist:
                        min_dist = dist
                        best_match_id = lost_id

                # If a match is found, re-assign the ID
                if best_match_id:
                    print(f"✅ Re-linking: New detection {new_id} -> Lost track {best_match_id}")
                    # Revive the old ball object
                    tracked_balls[best_match_id] = Ball(best_match_id, lost_tracks[best_match_id]['last_pos'])
                    del lost_tracks[best_match_id]
                    # Update the ID in this frame's list
                    idx = final_track_ids.index(new_id)
                    final_track_ids[idx] = best_match_id
                    # Update the boxes map with the correct ID
                    boxes_map[best_match_id] = boxes_map.pop(new_id)


            # 4. Update Ball Objects
            for i, track_id in enumerate(final_track_ids):
                centroid = ( (raw_boxes[i][0] + raw_boxes[i][2]) / 2.0, (raw_boxes[i][1] + raw_boxes[i][3]) / 2.0 )
                if track_id not in tracked_balls:
                    tracked_balls[track_id] = Ball(track_id, centroid)
                else:
                    tracked_balls[track_id].update(centroid)

        # 5. Manage Lost Tracks
        final_track_ids_set = set(final_track_ids)
        lost_this_frame = set(tracked_balls.keys()) - final_track_ids_set
        for lost_id in lost_this_frame:
            lost_tracks[lost_id] = {'last_pos': tracked_balls[lost_id].last_known_position, 'frame_lost': frame_count}
            del tracked_balls[lost_id]

        # Clean up tracks that have been lost for too long
        for lost_id in list(lost_tracks.keys()):
            if frame_count - lost_tracks[lost_id]['frame_lost'] > MAX_LOST_FRAMES:
                del lost_tracks[lost_id]

        # 6. Update Ball States
        update_ball_states(tracked_balls, player_feet_centroids)

        # 7. Draw Annotations
        draw_annotations(annotated_frame, tracked_balls, boxes_map, pose_results, player_feet_centroids)

        writer.write(annotated_frame)

    # --- Cleanup & Finalization ---
    cap.release()
    writer.release()
    pose_model.close()
    print(f"✅ Processing complete. Final video saved to {OUTPUT_PATH}")

    # Display the final video
    # video_file = open(OUTPUT_PATH, "r+b").read()
    # video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
    # display(HTML(f"""<video width=700 controls><source src="{video_url}"></video>"""))

if __name__ == '__main__':
    main()